In [ ]:
using Pkg
using Random
using Statistics
using Printf
using LinearAlgebra
using Logging

function find_project_root(start::AbstractString=pwd())
    active_project = Base.active_project()
    if !isnothing(active_project)
        active_root = dirname(active_project)
        if isfile(joinpath(active_root, "Project.toml")) && isfile(joinpath(active_root, "src", "System1D.jl"))
            return active_root
        end
    end

    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()
Pkg.activate(PROJECT_ROOT; io=devnull)

using Revise
using Plots

includet(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "periodic_ion_ring_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

includet(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D
includet(joinpath(PATHS.notebook_dir, "periodic_ion_ring_helpers.jl"))

default(; dpi=170)
nothing


## Single Particle on a Periodic Ion Ring

This notebook builds a one-particle Hamiltonian on a ring with `M` equally spaced soft-Coulomb ions,
solves the periodic one-body problem in a real Fourier basis, and uses the lowest exact orbital as both
the guiding trial state and the VMC warm-start distribution for GFMC.

Because the trial state is the exact one-body ground state, this notebook is meant to be a direct validation
benchmark for the generic Hamiltonian + `ImportanceGuiding(trial, H)` + `run_gfmc_with_vmc_init(...)` path.


In [ ]:
M = 3
a = 1.0
D = 0.5
ion_strength = 1.0
ion_softening = 0.35 * a
kmax = 6
quad_points = 1536

ring = build_periodic_ion_ring_model(
    M,
    a;
    D=D,
    ion_strength=ion_strength,
    ion_softening=ion_softening,
    kmax=kmax,
    quad_points=quad_points,
)

H = periodic_ion_hamiltonian(ring, 1)
trial = single_particle_trial_wavefunction(ring; orbital_index=1)
guiding = ImportanceGuiding(trial, H)
exact_energy = ring.energies[1]

targetN = 2^13

vmc_dt = 1.0e-2
vmc_nsteps = 60
vmc_params = VMCParams(; dt=vmc_dt, nsteps=vmc_nsteps, targetN=targetN, ET0=exact_energy)

gfmc_dt = 1.0e-3
gfmc_nsteps = 3000
gfmc_nequil = 60
feedback = 1
reconfiguration_interval = 2
branch_cap = 5.0
energy_window = 20
gfmc_params = GFMCParams(gfmc_dt, gfmc_nsteps, gfmc_nequil, targetN, exact_energy, feedback, reconfiguration_interval, branch_cap, energy_window)

rng_init = MersenneTwister(1234)
initial_positions = sample_uniform_ring_configurations(1, ring.L, targetN, rng_init)

MODEL_GRID_POINTS = 600
xgrid_model = Float64[i * (ring.L / MODEL_GRID_POINTS) for i in 0:(MODEL_GRID_POINTS - 1)]
onebody_potential_curve = Float64[onebody_potential(ring, x) for x in xgrid_model]
phi0_curve, phi0_grad_curve, phi0_lapl_curve = orbital_curve(ring, 1, xgrid_model)
dx_model = ring.L / MODEL_GRID_POINTS
onebody_trial_density = phi0_curve .^ 2
onebody_trial_density ./= (sum(onebody_trial_density) * dx_model)

SNAPSHOT_STEPS = [0, 1, 500, 1200]
DENSITY_GRID_POINTS = 400
DENSITY_BANDWIDTH = 0.08 * a
PERIOD_MARKERS = collect(0.0:a:ring.L)

RUN_LABEL = "guided warm-started single particle"
RUN_COLOR = :navy
PLOT_TITLE = "Periodic ion ring GFMC"
MODEL_TRIAL_TITLE = "Single-particle periodic ion ring diagnostics"
DENSITY_TITLE = "Periodic ion ring GFMC: one-body density evolution"

VMC_PROPOSAL = DriftGaussianProposal()
RECONFIGURATION = SystematicReconfiguration()

VMC_SHOW_PROGRESS = false
VMC_PROGRESS_EVERY = 0
VMC_DEBUG_MODE = false
VMC_DEBUG_EVERY = 10

SHOW_PROGRESS = true
PROGRESS_EVERY = 1
DEBUG_MODE = false
DEBUG_EVERY = 20

WRITE_RUN_CSV = false
CSV_FILENAME = "periodic_ion_ring_single_particle_gfmc_vmc_init.csv"
SAVE_FIGURES = false
FIGURE_STEM = "periodic_ion_ring_single_particle_gfmc_vmc_init"


In [ ]:
sim = run_gfmc_with_vmc_init(
    H,
    gfmc_params,
    initial_positions,
    trial,
    vmc_params;
    vmc_rng=MersenneTwister(41),
    gfmc_rng=MersenneTwister(52),
    proposal=VMC_PROPOSAL,
    guiding=guiding,
    nodepolicy=NoNode(),
    reconfiguration=RECONFIGURATION,
    vmc_show_progress=VMC_SHOW_PROGRESS,
    vmc_progress_every=VMC_PROGRESS_EVERY,
    vmc_progress_label="VMC warm start",
    vmc_debug=VMC_DEBUG_MODE,
    vmc_debug_every=VMC_DEBUG_EVERY,
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(gfmc_params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

println("GFMC step 0 corresponds to the VMC warm-start ensemble.")
println(@sprintf("exact one-body ground energy = %.8f", exact_energy))
println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, gfmc_params.nequil, mean_energy, sem_energy))
println(@sprintf("post-equilibration energy error = %.3e", mean_energy - exact_energy))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
step0_snapshot = sim.walker_positions_history[1]
step0_xs = nb_snapshot_coordinate(step0_snapshot, 1)
step0_centers, step0_density = nb_periodic_kde_curve(
    step0_xs;
    xmin=0.0,
    xmax=ring.L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

final_snapshot = nb_last_snapshot(sim)
final_xs = nb_snapshot_coordinate(final_snapshot, 1)
final_centers, final_density = nb_periodic_kde_curve(
    final_xs;
    xmin=0.0,
    xmax=ring.L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

p_potential = plot(
    xgrid_model,
    onebody_potential_curve;
    xlabel="x",
    ylabel="V(x)",
    title="Periodic ion lattice potential",
    color=:black,
    linewidth=2.4,
    label="V_latt(x)",
    xlims=(0.0, ring.L),
)
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_potential, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

p_orbital = plot(
    xgrid_model,
    phi0_curve;
    xlabel="x",
    ylabel="amplitude",
    title="Lowest exact orbital",
    color=:teal,
    linewidth=2.4,
    label="ϕ₀(x)",
    xlims=(0.0, ring.L),
)
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_orbital, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

p_density_compare = plot(
    xlabel="x",
    ylabel="density",
    title="Warm-start and GFMC density vs exact |ϕ₀|²",
    legend=:topright,
    xlims=(0.0, ring.L),
)
plot!(p_density_compare, xgrid_model, onebody_trial_density; color=:black, linewidth=2.2, linestyle=:dash, label="exact |ϕ₀|²")
plot!(p_density_compare, step0_centers, step0_density; color=RUN_COLOR, linewidth=2.4, label="step 0 density (after VMC)")
plot!(p_density_compare, final_centers, final_density; color=:crimson, linewidth=2.2, linestyle=:dot, label="final GFMC density")
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_density_compare, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

model_fig = plot(p_potential, p_orbital, p_density_compare; layout=(3, 1), size=(1200, 1100), plot_title=MODEL_TRIAL_TITLE)
display(model_fig)
nb_save_figure(model_fig, PATHS.figures_dir, FIGURE_STEM, "model_trial"; enabled=SAVE_FIGURES)

history_fig = nb_plot_gfmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(sim.walker_positions_history))]
density_fig = plot(
    xlabel="x",
    ylabel="density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, ring.L),
)
for (snapshot, step_idx) in zip(sim.walker_positions_history, available_steps)
    xs = nb_snapshot_coordinate(snapshot, 1)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=ring.L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    step_label = step_idx == 0 ? "step 0 (after VMC warm start)" : "step $(step_idx)"
    plot!(density_fig, centers, density; label=step_label, color=RUN_COLOR, linewidth=2.2, alpha=0.82)
end
plot!(density_fig, xgrid_model, onebody_trial_density; color=:black, linewidth=2.0, linestyle=:dash, label="exact |ϕ₀|²")
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "ion cell markers" : ""))
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)
